# Whisper Evaluation

Test whether the data is usable for Whisper fine-tuning. Trains on a ~2h stratified sample and evaluates if the model learns.

- **Model:** `openai/whisper-small`
- **GPU:** T4 (Colab)
- **Language:** Macedonian (`mk`)
- **Data:** 30s .wav chunks with Cyrillic transcriptions

In [1]:
!pip install -q --upgrade transformers datasets accelerate evaluate jiwer librosa

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 98.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 36.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 21.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 127.1 MB/s eta 0:00:00


In [2]:
import os
import json
import random
import librosa
import evaluate
import torch

from pathlib import Path
from collections import Counter, defaultdict
from dataclasses import dataclass
from typing import Any

from google.colab import drive
from datasets import Dataset, DatasetDict
from transformers import WhisperForConditionalGeneration, WhisperProcessor, Seq2SeqTrainingArguments, Seq2SeqTrainer

## Mount Drive & Discover Data

Walks the tree, collects all chunks from `training_data.jsonl` files, and reads dialect labels from `metadata.json`.

In [3]:
drive.mount("/content/drive")

Mounted at /content/drive


In [4]:
CORPUS_ROOT = Path("/content/drive/MyDrive/Vezilka/govori")
assert CORPUS_ROOT.exists(), f"Not found: {CORPUS_ROOT}"

video_dirs = []
for dirpath, _, filenames in os.walk(CORPUS_ROOT):
    if "training_data.jsonl" in filenames and "metadata.json" in filenames:
        video_dirs.append(Path(dirpath))
video_dirs.sort()

all_chunks = []
govor_counts = Counter()

for vdir in video_dirs:
    with open(vdir / "metadata.json", "r", encoding="utf-8") as f:
        metadata = json.load(f)
    govor = metadata[0].get("govor", "unknown") if metadata else "unknown"

    with open(vdir / "training_data.jsonl", "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            entry = json.loads(line)
            all_chunks.append({
                "audio_path": str(vdir / entry["audio"].replace("\\", "/")),
                "text": entry["text"],
                "govor": govor,
            })
            govor_counts[govor] += 1

print(f"Video directories: {len(video_dirs)}")
print(f"Total chunks:      {len(all_chunks)}")
print(f"Estimated audio:   ~{len(all_chunks) * 30 / 3600:.1f}h")
print()
print("Chunks per govor:")
for govor, count in govor_counts.most_common():
    print(f"  {govor:25s} {count:5d}  (~{count * 30 / 3600:.1f}h)")

Video directories: 97
Total chunks:      12861
Estimated audio:   ~107.2h

Chunks per govor:
  literaturen                7001  (~58.3h)
  skopski                    3720  (~31.0h)
  bitolski                    521  (~4.3h)
  unknown                     326  (~2.7h)
  kumanovski                  218  (~1.8h)
  ohridski                    195  (~1.6h)
  kochanski                   137  (~1.1h)
  strushki                    135  (~1.1h)
  prilepski                   121  (~1.0h)
  gevgeliski                  118  (~1.0h)
  shtipski                    116  (~1.0h)
  veleshki                    102  (~0.8h)
  kichevsko-porechki           92  (~0.8h)
  ovchepolski                  59  (~0.5h)


## Sample & Split

Stratified sample of ~250 chunks (~2h) proportional to each dialect's share. Split 80/20 into train and eval.

In [5]:
TARGET_CHUNKS = 250
SEED = 42
random.seed(SEED)

In [6]:
by_govor = defaultdict(list)
for chunk in all_chunks:
    by_govor[chunk["govor"]].append(chunk)

sampled = []
for govor, chunks in by_govor.items():
    n = max(1, round(TARGET_CHUNKS * len(chunks) / len(all_chunks)))
    n = min(n, len(chunks))
    sampled.extend(random.sample(chunks, n))

random.shuffle(sampled)

split_idx = int(len(sampled) * 0.8)
train_data = sampled[:split_idx]
eval_data = sampled[split_idx:]

print(f"Sampled: {len(sampled)} chunks (~{len(sampled) * 30 / 3600:.1f}h)")
print(f"Train:   {len(train_data)}")
print(f"Eval:    {len(eval_data)}")
print()

all_govori = sorted(set(c["govor"] for c in sampled))
train_g = Counter(c["govor"] for c in train_data)
eval_g = Counter(c["govor"] for c in eval_data)

print(f"{'govor':25s} {'train':>6s} {'eval':>6s}")
print("-" * 40)
for g in all_govori:
    print(f"{g:25s} {train_g.get(g, 0):6d} {eval_g.get(g, 0):6d}")

dataset = DatasetDict({
    "train": Dataset.from_list(train_data),
    "eval": Dataset.from_list(eval_data),
})
print()

Sampled: 249 chunks (~2.1h)
Train:   199
Eval:    50

govor                      train   eval
----------------------------------------
bitolski                       7      3
gevgeliski                     1      1
kichevsko-porechki             2      0
kochanski                      2      1
kumanovski                     3      1
literaturen                  109     27
ohridski                       4      0
ovchepolski                    1      0
prilepski                      2      0
shtipski                       2      0
skopski                       56     16
strushki                       2      1
unknown                        6      0
veleshki                       2      0



In [7]:
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['audio_path', 'text', 'govor'],
        num_rows: 199
    })
    eval: Dataset({
        features: ['audio_path', 'text', 'govor'],
        num_rows: 50
    })
})


## Load Model & Preprocess

Load `whisper-small` and process all audio into log-mel spectrograms with tokenized Cyrillic labels.

In [8]:
MODEL_NAME = "openai/whisper-small"

processor = WhisperProcessor.from_pretrained(
    MODEL_NAME, language="mk", task="transcribe"
)
model = WhisperForConditionalGeneration.from_pretrained(MODEL_NAME)
model.generation_config.language = "mk"
model.generation_config.task = "transcribe"
model.generation_config.forced_decoder_ids = None


def prepare_batch(batch):
    audio, _ = librosa.load(batch["audio_path"], sr=16000)
    batch["input_features"] = processor.feature_extractor(
        audio, sampling_rate=16000
    ).input_features[0]
    batch["labels"] = processor.tokenizer(batch["text"]).input_ids
    return batch


print("Preprocessing dataset (loading audio from Drive)...")
dataset = dataset.map(prepare_batch, remove_columns=dataset["train"].column_names)
print("Done.")
print(dataset)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:85: UserWarning: 
Access to the secret `HF_TOKEN` has not been granted on this notebook.
You will not be requested again.
Please restart the session if you want to be prompted again.
  warnings.warn(


preprocessor_config.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

generation_config.json: 0.00B [00:00, ?B/s]

Preprocessing dataset (loading audio from Drive)...


Map:   0%|          | 0/199 [00:00<?, ? examples/s]

Map:   0%|          | 0/50 [00:00<?, ? examples/s]

Done.
DatasetDict({
    train: Dataset({
        features: ['input_features', 'labels'],
        num_rows: 199
    })
    eval: Dataset({
        features: ['input_features', 'labels'],
        num_rows: 50
    })
})


## Training

Fine-tune for 500 steps. If the loss drops, the data works.

In [9]:
wer_metric = evaluate.load("wer")


@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(self, features):
        input_features = [{"input_features": f["input_features"]} for f in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        label_features = [{"input_ids": f["labels"]} for f in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1), -100
        )
        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels
        return batch


def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id
    pred_str = processor.tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)
    return {"wer": wer_metric.compute(predictions=pred_str, references=label_str)}


training_args = Seq2SeqTrainingArguments(
    output_dir="/content/whisper-training",
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=1e-5,
    warmup_steps=50,
    max_steps=500,
    fp16=True,
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,
    logging_steps=25,
    predict_with_generate=True,
    generation_max_length=225,
    report_to="none",
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["eval"],
    data_collator=DataCollatorSpeechSeq2SeqWithPadding(processor=processor),
    compute_metrics=compute_metrics,
    processing_class=processor,
)

trainer.train()

Step,Training Loss,Validation Loss,Wer
100,1.100237,1.272607,0.844846
200,0.257949,1.479365,0.809768
300,0.045427,1.532826,0.736373
400,0.007826,1.607691,0.775769
500,0.005934,1.624826,0.767944


[transformers] The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transform

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['proj_out.weight'].


TrainOutput(global_step=500, training_loss=0.5426238204836845, metrics={'train_runtime': 2502.6531, 'train_samples_per_second': 3.197, 'train_steps_per_second': 0.2, 'total_flos': 2.20998699565056e+18, 'train_loss': 0.5426238204836845, 'epoch': 38.48})

## Evaluation

Run the trained model on eval samples. If WER is reasonable and transcriptions look coherent, the data pipeline works.

In [10]:
model.eval()
results = []

for sample in eval_data:
    audio, _ = librosa.load(sample["audio_path"], sr=16000)
    inputs = processor.feature_extractor(
        audio, sampling_rate=16000, return_tensors="pt"
    ).input_features.to(model.device)

    with torch.no_grad():
        pred_ids = model.generate(inputs)

    pred_text = processor.tokenizer.batch_decode(pred_ids, skip_special_tokens=True)[0]
    results.append({"govor": sample["govor"], "truth": sample["text"], "pred": pred_text})

overall_wer = wer_metric.compute(
    predictions=[r["pred"] for r in results],
    references=[r["truth"] for r in results],
)
print(f"Overall WER: {overall_wer:.2%}")
print()

by_govor_results = defaultdict(lambda: {"preds": [], "refs": []})
for r in results:
    by_govor_results[r["govor"]]["preds"].append(r["pred"])
    by_govor_results[r["govor"]]["refs"].append(r["truth"])

print(f"{'govor':25s} {'WER':>8s} {'samples':>8s}")
print("-" * 45)
for g in sorted(by_govor_results):
    g_wer = wer_metric.compute(
        predictions=by_govor_results[g]["preds"],
        references=by_govor_results[g]["refs"],
    )
    print(f"{g:25s} {g_wer:>7.2%} {len(by_govor_results[g]['preds']):>8d}")

print()
print("=" * 100)
print("Sample transcriptions (first 15):")
print("=" * 100)
for i, r in enumerate(results[:15]):
    print()
    print(f"--- Sample {i} [{r['govor']}] ---")
    print(f"  TRUTH: {r['truth'][:150]}")
    print(f"  PRED:  {r['pred'][:150]}")
print()
print("=" * 100)

Overall WER: 94.55%

govor                          WER  samples
---------------------------------------------
bitolski                  198.99%        3
gevgeliski                 99.12%        1
kochanski                 400.00%        1
kumanovski                 80.68%        1
literaturen                80.41%       27
skopski                    99.64%       16
strushki                   73.44%        1

Sample transcriptions (first 15):

--- Sample 0 [literaturen] ---
  TRUTH: знаевте дека во буџетот немате доволно пари. Во меѓувреме има одлука на Комисијата за заштита од дискриминација. Потврдуваат дека е сторена дискримина
  PRED:  во мене се

--- Sample 1 [bitolski] ---
  TRUTH: емисија во Прилеп и зелка дробевме во југото и реков уште зелка немам дробено во југото. Уште се немаш забришувано. Уште се немам и забришувано. Ако н
  PRED:   Епуктеш леле господе златен кога не се секирај. А може што не се немам изобришува на потуре. Након и застани полиција сега на како да им објас